# Notebook 01 — Production Structure Pipeline

Notebook này có một pipeline production duy nhất: tự resolve/restore Phase00, restore checkpoint theo `video_id + stage`, xử lý tuần tự từng video, package và sync Phase01. Chỉ sửa cell `USER SETTINGS`, sau đó **Run All**.

Phase01 v1.2 dùng NVIDIA FastConformer cho ASR, gate ảnh không có chữ trước Vintern-1B-v3_5 (OCR), rồi dùng một Vintern-3B-R-beta fp16 xuyên caption, scene boundary và scene summary. Gemini chỉ là fallback tùy chọn khi model local lỗi.

Business logic và cấu hình deterministic nằm trong `src/system1/` và `configs/`. Local/Colab/Kaggle chỉ là scratch/runtime; checkpoint Hugging Face có quyền ghi (public hoặc private) là resume authority. Lưu ý: checkpoint public cũng làm các artifact trung gian có thể được đọc công khai.

Repo được lấy từ GitHub bằng `git clone` lần đầu và `git fetch` + fast-forward ở lần sau (`git pull --ff-only` tương đương). Package được cài bằng `pip install -e` với extra production. Các root tương thích orchestration gồm `AIC_REPO_ROOT`, `AIC_REPO_PARENT`, `AIC_DATA_ROOT`, `AIC_RUNTIME_ROOT`, và `AIC_ARTIFACT_ROOT`.

In [ ]:
# USER SETTINGS — teammate chỉ sửa các giá trị trong cell này, không sửa các cell pipeline bên dưới.
# Mỗi người nhận đúng batch do Notebook 00B tạo tại <release>/manifests/batch_000.txt, batch_001.txt, ...
# Không tự đặt batch_id mới: tên không khớp manifest sẽ làm worker không tìm thấy danh sách video.
batch_id = "batch_000"


# worker_id dùng cho report/log/checkpoint ownership. Các worker chạy song song phải dùng ID khác nhau.
# Khuyến nghị ghép cùng số với batch để dễ audit: batch_003 -> worker_003.
worker_id = "worker_000"


# Thường giữ None để chọn release Phase00 hoàn tất mới nhất theo completed_at.
# AIOU26_release hiện có completed_at nên giữ None để auto-resolve. Chỉ override khi cả team muốn pin một release cụ thể.
release_id_override = None

# Kaggle cap 2x T4. True = chia doi danh sach video, moi GPU mot tien trinh.
# Dat False neu phien chi co 1 GPU hoac muon chay tuan tu nhu truoc.
use_dual_gpu = True

# HF STORAGE — giữ None để dùng versioned default trong configs/storage.yaml:
#   release: 1thesudden/AIOU26_release; checkpoint: 1thesudden/AIOU26_checkpoints.
# Chỉ đặt AIOU26_release_test khi repo test đã tồn tại và HF_TOKEN có quyền ghi. Không dùng tên AIC2026/AIC26 cũ.
hf_release_repo = None    # Dùng 1thesudden/AIOU26_release từ config; mọi worker trong cùng run phải trỏ cùng repo.
hf_checkpoint_repo = None    # State/artifact resume; public hoặc private, nhưng HF_TOKEN bắt buộc có quyền ghi.

# revision/prefix là scope nâng cao. Giữ None nếu không có kế hoạch namespace riêng cho cả team.
hf_revision = None
hf_prefix = None
checkpoint_revision = None
checkpoint_prefix = None  # Chỉ tách checkpoint TEST khi cả team dùng cùng namespace riêng.

# ASR: None hoặc "nemo" = NVIDIA FastConformer/Parakeet mặc định.
# Chỉ đặt "faster_whisper" để dùng Systran/faster-whisper-large-v3 thay thế.
asr_provider = None
# Scratch chứa video/model/artifact tạm và có thể rất lớn. None chọn /content hoặc /kaggle/temp an toàn hơn.
# Nếu override, dùng ổ local nhanh còn ít nhất 35 GiB; không trỏ vào Google Drive hoặc thư mục output được auto-save.
scratch_dir_override = None

# SOURCE CODE — giữ nguyên cho teammate. Run chính thức dùng dev; không dùng branch cá nhân khác nhau giữa worker.
github_repo_url = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
github_branch = "dev"
repo_dir_name = "Multimodal-Agentic-Retrieval-Engine"

# Secrets không đặt trực tiếp trong notebook. HF_TOKEN bắt buộc để restore/sync.
# GEMINI_API_KEY là tùy chọn; chỉ được dùng nếu Qwen cần fallback.


## Cách teammate tự setting để chạy Notebook 01

Chỉ cần sửa cell `USER SETTINGS` phía trên, sau đó chọn **Run All**.

`batch_id` là batch mà máy/người này sẽ xử lý. Giá trị này phải khớp với file do Notebook 00B đã tạo trong `manifests/`, ví dụ `batch_000.txt`, `batch_001.txt`, ...

`worker_id` là tên worker dùng để ghi log/report. Nên đặt trùng số với batch cho dễ theo dõi.

Ví dụ nếu Notebook 00B tạo 10 batch:

```text
Người 1:  batch_id = "batch_000", worker_id = "worker_000"
Người 2:  batch_id = "batch_001", worker_id = "worker_001"
Người 3:  batch_id = "batch_002", worker_id = "worker_002"
...
Người 10: batch_id = "batch_009", worker_id = "worker_009"
```

`release_id_override`: giữ `None` cho `AIOU26_release` hiện tại vì manifest đã có `completed_at` và auto-resolve được `canonical_release_v001`. Chỉ đặt chuỗi release cụ thể khi cả team chủ động pin cùng một snapshot.

`hf_release_repo = None` và `hf_checkpoint_repo = None` nghĩa là dùng default trong `configs/storage.yaml`:

```text
release repo:    1thesudden/AIOU26_release
checkpoint repo: 1thesudden/AIOU26_checkpoints
```

Notebook 01 không đọc trực tiếp raw repo. Raw repo của Notebook 00B là `1thesudden/AIOU26_raw`.

Nếu chỉ chạy thử với repo test, chỉ đặt `hf_release_repo = "1thesudden/AIOU26_release_test"` khi repo đó đã tồn tại và token có quyền đọc/ghi. Nếu không chắc, giữ `None` để dùng `1thesudden/AIOU26_release`.

Mỗi teammate phải cấu hình secret trước khi chạy:

```text
HF_TOKEN
GEMINI_API_KEY  # optional fallback
```

`HF_TOKEN` phải có quyền ghi vào Hugging Face release repo và checkpoint repo, vì Notebook 01 chạy với `--sync` để upload kết quả Phase01.

Model local/free trong Notebook 01:

```text
OCR:          5CD-AI/Vintern-1B-v3_5
ASR:          nvidia/parakeet-ctc-0.6b-vi (NeMo/FastConformer)
Shot caption: Qwen/Qwen2.5-VL-7B-Instruct 4-bit NF4
Scene boundary/summary: cùng Qwen đã load trong runtime chunk
Semantic fallback: Gemini nếu có GEMINI_API_KEY
```

Free/local nghĩa là model chạy trên GPU của laptop, Colab free hoặc Kaggle free; luồng mặc định không tốn Gemini API nhưng vẫn cần đủ VRAM, runtime disk và thời gian tải model. Ảnh đưa vào VLM là representative keyframe của shot; output lưu thêm `ocr.parquet` và `shot_captions.parquet` schema v2 gồm caption, objects, actions, visible text summary.

Runtime chunking được package tự động điều chỉnh theo dung lượng scratch còn trống (tối đa 4 video và khoảng 1.5 GiB raw mỗi chunk). Teammate không tự chia nhỏ `batch_id`; cứ chọn batch do Notebook 00B tạo và chạy **Run All**. OCR request batch mặc định là 4, caption batch là 2 và tự giảm về 1 khi OOM; scene boundary/summary giữ batch 1. Vintern unload trước khi Qwen load, và Qwen chỉ load một lần cho mỗi runtime chunk.

Trên Kaggle, notebook dùng `/kaggle/temp/aic_phase01` cho model cache/scratch để tránh quota auto-save 20GB của `/kaggle/working`. Kết quả chính vẫn được sync lên Hugging Face.

`GEMINI_API_KEY` không bắt buộc cho run local-first. Nếu không cấu hình key và Qwen gặp lỗi cần fallback thì video đó sẽ fail rõ ràng thay vì silently đổi model.

Không sửa `github_repo_url`, `github_branch`, `repo_dir_name`, `asr_provider`, `scratch_dir_override` nếu không có lý do cụ thể.

In [ ]:
# BƯỚC 1: Detect environment, paths và secrets. Không in secret.
import os, shutil, sys
from pathlib import Path

if "google.colab" in sys.modules:
    runtime_env = "colab"
    workspace = Path("/content/aic_phase01")
elif "KAGGLE_URL_BASE" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    runtime_env = "kaggle"
    workspace = Path("/kaggle/temp/aic_phase01")
else:
    runtime_env = "local"
    workspace = Path.cwd() / ".phase01_runtime"
workspace.mkdir(parents=True, exist_ok=True)
output_root = workspace / "output"
scratch_dir = Path(scratch_dir_override).expanduser() if scratch_dir_override else workspace / "scratch"
model_cache = workspace / "model_cache"
for path in (output_root, scratch_dir, model_cache): path.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(workspace).free / (1024**3)
if runtime_env in {"colab", "kaggle"} and free_gb < 35:
    raise RuntimeError(f"Không đủ runtime disk cho model local mặc định: {free_gb:.1f} GiB trống < 35 GiB.")

def load_secret(name):
    if os.environ.get(name): return os.environ[name]
    if runtime_env == "colab":
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    if runtime_env == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return None

for secret_name in ("HF_TOKEN", "GEMINI_API_KEY"):
    value = load_secret(secret_name)
    if value: os.environ[secret_name] = value
if os.environ.get("HF_TOKEN"): os.environ["AIC_HF_TOKEN"] = os.environ["HF_TOKEN"]
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Cần cấu hình HF_TOKEN trong Colab/Kaggle Secrets hoặc environment.")

os.environ["HF_HOME"] = str(model_cache / "hf")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
os.environ.setdefault("HF_XET_SHARD_CACHE_SIZE_LIMIT", str(1024**3))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["AIC_DATA_ROOT"] = str(workspace / "data")
os.environ["AIC_RUNTIME_ROOT"] = str(workspace / "runtime")
os.environ["AIC_ARTIFACT_ROOT"] = str(workspace / "artifacts")
print({"environment": runtime_env, "workspace": str(workspace), "output_root": str(output_root), "scratch": str(scratch_dir)})

In [ ]:
# BƯỚC 2: Clone/update đúng branch GitHub mà không xóa thay đổi local.
import subprocess

def run_command(command, cwd=None):
    print("RUN:", " ".join(map(str, command)))
    result = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode: raise RuntimeError(f"Command failed ({result.returncode}): {command}")
    return result

def is_repo_root(path): return (path / "system1" / "src" / "system1" / "__init__.py").is_file()
repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if is_repo_root(path)), None)
if repo_root is None:
    repo_root = workspace / repo_dir_name
    if not (repo_root / ".git").exists():
        run_command(["git", "clone", "--branch", github_branch, "--single-branch", github_repo_url, str(repo_root)], cwd=workspace)
if run_command(["git", "status", "--porcelain"], cwd=repo_root).stdout.strip():
    raise RuntimeError("Repo có thay đổi local; notebook không tự reset/stash.")
run_command(["git", "fetch", "origin", github_branch], cwd=repo_root)
local_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", f"refs/heads/{github_branch}"], cwd=repo_root).returncode == 0
run_command(["git", "switch", github_branch] if local_branch else ["git", "switch", "--track", "-c", github_branch, f"origin/{github_branch}"], cwd=repo_root)
run_command(["git", "merge", "--ff-only", f"origin/{github_branch}"], cwd=repo_root)
os.environ["AIC_REPO_ROOT"] = str(repo_root)
os.environ["AIC_REPO_PARENT"] = str(repo_root.parent)

In [ ]:
# BƯỚC 3: Cài production dependencies và xác nhận kernel import đúng source.
system1_root = repo_root / "system1"
run_command([sys.executable, "-m", "pip", "install", "-q", "-e", f"{system1_root}[phase01-production]"])
source_root = str(system1_root / "src")
sys.path = [source_root] + [item for item in sys.path if item != source_root]
for name in list(sys.modules):
    if name == "system1" or name.startswith("system1."): del sys.modules[name]
import system1
import torch
actual = Path(system1.__file__).resolve()
expected = (system1_root / "src" / "system1" / "__init__.py").resolve()
if actual != expected: raise RuntimeError(f"Import source mismatch: actual={actual}, expected={expected}")
if runtime_env in {"colab", "kaggle"} and not torch.cuda.is_available():
    raise RuntimeError("Notebook 01 local VLM default cần GPU. Bật GPU runtime trước khi Run All.")
if torch.cuda.is_available():
    print({"cuda": True, "gpu": torch.cuda.get_device_name(0), "workspace_free_gb": round(shutil.disk_usage(workspace).free / (1024**3), 2)})
print("package source preflight: OK", actual)

In [ ]:
# BƯỚC 4: Helper CLI có streaming output và error tail.
def run_cli(arguments):
    command = [sys.executable, "-m", "system1.cli", *map(str, arguments)]
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(command, cwd=system1_root, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True); lines.append(line)
    code = process.wait()
    if code:
        raise RuntimeError(f"CLI failed ({code}). Last output:\n" + "".join(lines[-120:]))
    return "".join(lines)

In [ ]:
# BƯỚC 5: Một lệnh production — auto-resolve, restore, preflight, resume, package, sync.
command = [
    "process-batch", "--batch-id", batch_id, "--worker-id", worker_id,
    "--output", str(output_root), "--scratch-dir", str(scratch_dir),
    "--require-frame-timeline", "--restore-phase00", "--validate-remote", "--sync",
]
optional = {
    "--asr-provider": asr_provider,
    "--release-id-override": release_id_override,
    "--hf-repo-id": hf_release_repo,
    "--hf-checkpoint-repo": hf_checkpoint_repo,
    "--hf-revision": hf_revision,
    "--hf-prefix": hf_prefix,
    "--checkpoint-revision": checkpoint_revision,
    "--checkpoint-prefix": checkpoint_prefix,
}
for option, value in optional.items():
    if value not in (None, ""): command.extend([option, str(value)])
if use_dual_gpu:
    # Chia manifest roi goi lai process-batch mot lan cho moi GPU.
    # Script tu tim release_id tu thu muc output.
    dual_script = repo_root / "system1" / "scripts" / "run_dual_gpu.py"
    run_cli_dual = [sys.executable, str(dual_script),
                    "--batch-id", batch_id, "--worker-id", worker_id,
                    "--output", str(output_root),
                    # bo 7 phan tu dau: script tu truyen batch/worker/output
                    *command[7:]]
    subprocess.run(run_cli_dual, check=True)
else:
    run_cli(command)


In [ ]:
# BƯỚC 6: Báo cáo ngắn sau Run All.
import json
last_run = json.loads((output_root / "phase01_last_run.json").read_text())
release_root = Path(last_run["release_dir"])
resolved = json.loads((release_root / "manifests/phase01/resolved_config.json").read_text())
report_path = release_root / "manifests/worker_reports" / f"structure_{batch_id}_{worker_id}.json"
review_path = release_root / "manifests/phase01" / f"manual_review_{batch_id}_{worker_id}.json"
report = json.loads(report_path.read_text())
review = json.loads(review_path.read_text())
print({
    "release_id": resolved["runtime"]["release_id"],
    "config_hash": resolved["config_hash"],
    "counts": report.get("counts"),
    "manual_review_status": review["status"],
    "manual_review_samples": review["sample_size_actual"],
    "report": str(report_path),
})

In [ ]:
# BƯỚC 7: Verify trực tiếp output Phase01 đã hiện diện đầy đủ trên Hugging Face.
from huggingface_hub import HfApi

release_storage = resolved["storage"]["release"]
release_id = resolved["runtime"]["release_id"]
remote_root = f"{release_id}/phase01_structure"
prefix = str(release_storage.get("prefix") or "").strip("/")
scoped_root = f"{prefix}/{remote_root}" if prefix else remote_root
api = HfApi(token=os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN"))
entries = api.list_repo_tree(
    repo_id=release_storage["repo_id"],
    repo_type=release_storage.get("repo_type", "dataset"),
    revision=release_storage.get("revision", "main"),
    path_in_repo=scoped_root,
    recursive=True,
)
remote_files = {entry.path for entry in entries if getattr(entry, "path", None)}
complete_videos = [row["video_id"] for row in report.get("videos", []) if str(row.get("status", "")).startswith("complete")]
package_root = resolved["artifact"]["package"]["root"].format(
    release_id=release_id, batch_id=batch_id, video_id=""
).strip("/")
package_filename = resolved["artifact"]["package"]["filename"]
expected = {f"{prefix}/{package_root}/{package_filename.format(video_id=video_id)}" if prefix else f"{package_root}/{package_filename.format(video_id=video_id)}" for video_id in complete_videos}
expected.update({
    f"{scoped_root}/worker_reports/structure_{batch_id}_{worker_id}.json",
    f"{scoped_root}/worker_reports/errors_{batch_id}_{worker_id}.jsonl",
    f"{scoped_root}/worker_reports/manual_review_{batch_id}_{worker_id}.json",
})
missing = sorted(expected - remote_files)
if missing: raise RuntimeError("HF Phase01 output verification failed; missing: " + ", ".join(missing))
print({
    "hf_repo": release_storage["repo_id"],
    "remote_root": scoped_root,
    "verified_files": len(expected),
    "verified_packages": len(complete_videos),
    "status": "HF phase01_structure verification: OK",
})
